In [1]:
#from mpl_toolkits import mplot3d
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split
from IPython.core.pylabtools import figsize
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from scipy import stats
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from scipy.stats import pearsonr,spearmanr
from itertools import combinations
from scipy.special import comb, perm
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import ShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

scaler_M = MinMaxScaler()
scaler_S = StandardScaler()

rs = ShuffleSplit(n_splits=100, test_size=.2, random_state=0)

plt.rcParams['savefig.dpi'] = 100 #图片像素
plt.rcParams['figure.dpi'] = 100#分辨率

In [3]:
data=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 4\select by pearson from initial-MA-clipped.csv")

F=np.array(data)
p=data.ev_ratio
p=np.array(p)
F_0=F[:,3:67]
sf=np.array(data.sf)/100
Vsv=np.array(data.sv_Volume)
Vso=np.array(data.so_Volume)
N_sv=np.array(data.N_sv)
N_so=np.array(data.N_so)

In [4]:
F_0.shape

(409, 64)

In [5]:
a=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_alpha_SBS from initial-MA-clipped-17.csv")
b=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_gamma_SBS from initial-MA-clipped-17.csv")

_alpha=np.array(a).flatten()
_gamma=np.array(b).flatten()

In [6]:
m=np.shape(F_0)[1]

m_r_2=[]
s_r_2=[]
    
m_r_2_tr=[]
s_r_2_tr=[]

for k in range(0,m):
        
    evaluate_r_2=[]
    evaluate_r_2_tr=[]
        
    n=-1
    
    F_c=F_0[:,k:k+1]
    
    for train_index, test_index in rs.split(F_c):
            
        scaler_M.fit(F_c[train_index,:])
            
        F_tr=scaler_M.transform(F_c[train_index,:])
        F_te=scaler_M.transform(F_c[test_index,:])
        
        n=n+1
        krr=KernelRidge(alpha=_alpha[n], 
                        #kernel='rbf',
                        kernel='laplacian',
                        gamma=_gamma[n])
        
        krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
            
        p_tr = krr.predict(F_tr)
        r_2_tr=r2_score((p.reshape(-1,1)[train_index,:]-1)*100, p_tr)
        evaluate_r_2_tr.append(r_2_tr)
    
        p_pre = krr.predict(F_te)   
        r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre)
        evaluate_r_2.append(r_2)
        
    m_r_2.append(np.mean(evaluate_r_2))
    s_r_2.append(np.std(evaluate_r_2))
    
    m_r_2_tr.append(np.mean(evaluate_r_2_tr))
    s_r_2_tr.append(np.std(evaluate_r_2_tr))
    
    r2_max=max(m_r_2)
    t=m_r_2.index(max(m_r_2))
    std_r2=s_r_2[t]
    
    r2_tr=m_r_2_tr[t]
    std_r2_tr=s_r_2_tr[t]

In [9]:
print(r2_tr)
print(std_r2_tr)

print(r2_max)
print(std_r2)

0.33707526266520177
0.024909523838467583
0.2951217437726603
0.10674231661542483


In [10]:
t

37

In [11]:
std_r2=[]
ind=[]
r2=[]

std_r2_tr=[]
r2_tr=[]

add=63

In [34]:
#SFS
z=t
F_c=F_0[:,z:z+1]
F_0=np.delete(F_0,z,1)

for l in range(0,add):
    
    m_r_2=[]
    s_r_2=[]
    
    m_r_2_tr=[]
    s_r_2_tr=[]
    
    for k in range(0,np.shape(F_0)[1]):
    
        F_f=F_0[:,k:k+1]
        F_s=np.concatenate((F_c, F_f), axis=1)
    
        evaluate_r_2=[]
        evaluate_r_2_tr=[]
        n=-1
    
        for train_index, test_index in rs.split(F_s):
            scaler_M.fit(F_s[train_index,:])
            
            F_tr=scaler_M.transform(F_s[train_index,:])
            F_te=scaler_M.transform(F_s[test_index,:])
            
            n=n+1
            krr=KernelRidge(alpha=_alpha[n], 
                            #kernel='rbf',
                            kernel='laplacian',
                            gamma=_gamma[n])
        
            krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
            
            p_tr = krr.predict(F_tr)
            r_2_tr=r2_score((p.reshape(-1,1)[train_index,:]-1)*100, p_tr)
            evaluate_r_2_tr.append(r_2_tr)
    
            p_pre = krr.predict(F_te)   
            r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre)
            evaluate_r_2.append(r_2)

        m_r_2.append(np.mean(evaluate_r_2))
        s_r_2.append(np.std(evaluate_r_2))
        
        m_r_2_tr.append(np.mean(evaluate_r_2_tr))
        s_r_2_tr.append(np.std(evaluate_r_2_tr))
        
    r2.append(max(m_r_2))
    t=m_r_2.index(max(m_r_2))
    std_r2.append(s_r_2[t])
    ind.append(t)
    
    r2_tr.append(m_r_2_tr[t])
    std_r2_tr.append(s_r_2_tr[t])
    
    F_c=np.concatenate((F_c, F_0[:,t:t+1]), axis=1)
    F_0=np.delete(F_0,t,1)

In [37]:
R=pd.DataFrame(r2_tr)
R.to_csv('r2_tr.csv')
R=pd.DataFrame(std_r2_tr)
R.to_csv('std_r2_tr.csv')

R=pd.DataFrame(r2)
R.to_csv('r2.csv')
R=pd.DataFrame(std_r2)
R.to_csv('std_r2.csv')

In [6]:
evaluate_r_2=[]
#sf
evaluatesf_r_2=[]

In [7]:
n=-1

for train_index, test_index in rs.split(F_0):
    
    scaler_M.fit(F_0[train_index,:])
    F_tr=scaler_M.transform(F_0[train_index,:])
    F_te=scaler_M.transform(F_0[test_index,:])
        
    n=n+1
            
    krr=KernelRidge(alpha=_alpha[n], 
                    kernel='rbf',
                    #kernel='laplacian',
                    gamma=_gamma[n])
        
    krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
    p_pre = krr.predict(F_te)
    
    #VLF
    r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre) 
    evaluate_r_2.append(r_2)
    #SF
    sf_pre=(((p_pre/100)+1)*Vso.reshape(-1,1)[test_index,:]-Vsv.reshape(-1,1)[test_index,:])/Vsv.reshape(-1,1)[test_index,:]
    r_2=r2_score(sf.reshape(-1,1)[test_index,:], sf_pre)

    evaluatesf_r_2.append(r_2)

In [8]:
#VLF
m_r_2=np.mean(evaluate_r_2)
s_r_2=np.std(evaluate_r_2)

print("VLF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))

#sf
m_r_2=np.mean(evaluatesf_r_2)
s_r_2=np.std(evaluatesf_r_2)

print("SF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))

VLF:
r_2_mean=0.65481
r_2_std=0.07521
MAE_mean=6.04942
MAE_std=0.67548
RMSE_mean=8.85485
RMSE_std=1.03777
per_mean=0.81561
per_std=0.04459
Ord_mean=0.80734
Ord_std=0.02689
SF:
r_2_mean=0.80481
r_2_std=0.06548
MAE_mean=7.23363
MAE_std=0.94289
RMSE_mean=11.44444
RMSE_std=2.09580
per_mean=0.90519
per_std=0.02982
Ord_mean=0.88562
Ord_std=0.01520
